# 01 — EDA (M2)

Base rate of breakthrough, class imbalance, coverage by birth year / academy / age bucket,
missingness of early-age features.

Loads `data/processed/features.parquet` if a full `run_ingest` has been done, otherwise the
**convenience-sample demo** (`features_demo.parquet`, built by `scripts/demo_dataset.py` from
current RPL squads via tmapi). The demo is target-skewed — few true negatives — so treat the
target distribution here as a pipeline check, not the real base rate.

In [ ]:
import sys

sys.path.insert(0, "..")
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

proc = Path("../data/processed")
path = proc / "features.parquet"
if not path.exists():
    path = proc / "features_demo.parquet"
df = pd.read_parquet(path)
print("loaded", path.name, df.shape)
from features.build_features import feature_columns

FEATS = feature_columns(df)
print(len(FEATS), "feature columns")

## Target distribution & base rate

In [ ]:
vc = (
    df["target"]
    .value_counts(dropna=False)
    .rename({1: "broke_through(1)", 0: "settled_no(0)", -1: "censored"})
)
print(vc)
resolved = df[df["target"] != -1]
if len(resolved):
    print(f"\nbase rate (of resolved): {resolved['target'].mean():.1%}")
    print(
        f"imbalance ratio 0:1 = {(resolved['target'] == 0).sum()}:{(resolved['target'] == 1).sum()}"
    )
print("\nordinal:", df["ordinal_target"].value_counts().to_dict())

## Coverage by birth year (drives the temporal split)

In [ ]:
by = df.groupby("birth_year").agg(
    n=("player_id", "size"),
    broke=("target", lambda s: (s == 1).sum()),
    censored=("target", lambda s: (s == -1).sum()),
)
print(by)
try:
    ax = by["n"].plot(kind="bar", figsize=(11, 3), title="players per birth-year cohort")
    ax.figure.tight_layout()
except Exception as e:
    print("plot skipped:", e)

## Age-bucket coverage

How many players actually have recorded minutes in each youth bucket. Expect U13/U15 to be
thin — structured data below ~U15 barely exists (SPEC §14).

In [ ]:
buckets = [c for c in df.columns if c.startswith("minutes_U")]
cov = pd.DataFrame(
    {
        "players_with_minutes": [(df[c] > 0).sum() for c in buckets],
        "median_minutes_if_any": [df.loc[df[c] > 0, c].median() for c in buckets],
    },
    index=buckets,
)
print(cov)
print("\nplayed_youth_league:", int(df["played_youth_league"].sum()), "/", len(df))

## Missingness of key features

In [ ]:
miss = df[FEATS].isna().mean().sort_values(ascending=False)
print((miss[miss > 0] * 100).round(1).astype(str) + " %")

## Do youth signals separate the classes? (resolved only)

In [ ]:
cols = [
    "youth_minutes_total",
    "youth_ga_per90",
    "youth_minutes_trend",
    "best_level_pre_cutoff",
    "market_value_at_cutoff_eur",
    "academy_conversion_rate",
]
cols = [c for c in cols if c in df.columns]
if len(resolved):
    print(resolved.groupby("target")[cols].mean().T)
else:
    print("no resolved rows in this sample")

## Notes for M3

- **Split**: temporal — train on cohorts resolved before `config[split].test_cohort_from`, test after.
- **Imbalance**: use PR-AUC / Recall@TopK, class weights; not accuracy.
- **`academy_conversion_rate`** is NaN for the earliest cohort of each academy by design (no prior history).
- Censored rows (`target == -1`) are dropped for the binary model, kept for survival (M5).